# mask

> Masking utilities used in measuring the ICL in images of galaxy clusters.

In [ ]:
# | default_exp mask

In [ ]:
# | exporti

import logging
from warnings import catch_warnings, filterwarnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from astropy.convolution import Gaussian2DKernel, convolve
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize
from photutils.segmentation import (
    deblend_sources,
    detect_sources,
    detect_threshold,
    SegmentationImage,
)
from photutils.utils import circular_footprint
from scipy.ndimage import binary_closing, binary_dilation, binary_erosion, label
from scipy.stats import median_abs_deviation

from nicl.background import get_background
from nicl.filter import sampled_median_filter, sampled_mad_filter
from nicl.main import configure_logging
from nicl.utilities import get_img_centre_pixel

In [ ]:
#  | exporti

configure_logging(__name__, level="DEBUG")

In [ ]:
# | export


def get_label_at_position(
    segm,  # a photutils SegmentationImage or numpy array mask
    image=None,  # the image on which the segmentation was performed
    position=None,  # position of the segment, if None assume the image centre
    wcs=None,  # WCS for converting a `position` supplied as a SkyCoord to pixels
):
    """Return the label of the segment at the specified position.

    The `position` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    If no `position` is provided, the image centre is assumed, in which case the `image` must be supplied.
    """
    if not isinstance(segm, SegmentationImage):
        segm = label(segm)
    if position is None:
        position = get_img_centre_pixel(image).astype(int)
    elif wcs is not None:
        position = tuple(int(x) for x in position.to_pixel(wcs))
    pixel_index = position[::-1]
    return segm.data[*pixel_index]


def keep_segment_at_position(
    segm,  # a photutils SegmentationImage or numpy array mask
    image=None,  # the image on which the segmentation was performed
    position=None,  # position of the segment, if None assume the image centre
    wcs=None,  # WCS for converting a `position` supplied as a SkyCoord to pixels
):
    """Return a mask containing just the segment at the specified position.

    The `position` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    If no `position` is provided, the image centre is assumed, in which case the `image` must be supplied.
    """
    label = get_label_at_position(segm, image, position, wcs)
    segm = segm.copy()
    segm.keep_label(label, relabel=True)
    return segm


def remove_segment_at_position(
    segm,  # a photutils SegmentationImage or numpy array mask
    image=None,  # the image on which the segmentation was performed
    position=None,  # position of the segment, if None assume the image centre
    wcs=None,  # WCS for converting a `position` supplied as a SkyCoord to pixels
):
    """Return a mask having removed the segment at the specified position.

    The `position` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    If no `position` is provided, the image centre is assumed, in which case the `image` must be supplied.
    """
    label = get_label_at_position(segm, image, position, wcs)
    segm = segm.copy()
    segm.remove_label(label, relabel=True)
    return segm

In [ ]:
# | export


def smooth_image(image, sigma):
    """Smooth `image` using Gaussian with standard deviation `sigma`."""
    kernel = Gaussian2DKernel(x_stddev=sigma, x_size=6 * sigma + 1)
    kernel.normalize()
    with catch_warnings():
        filterwarnings("ignore", ".*NaN values detected post convolution.*")
        smoothed = convolve(image.data, kernel)
    return smoothed

In [ ]:
# | export


def create_bcg_mask(
    image: np.ndarray,  # input image data to mask
    *,  # the following parameters must be provided as keyword arguments if required
    sb_threshold: float,  # surface brightness threshold for the BCG mask
    centre_pos=None,  # position of the centre of the BCG, if None assume the image centre
    wcs=None,  # WCS for converting a `centre_pos` supplied as a SkyCoord to pixels
):  # the BCG image mask
    """Create a BCG mask for the specified `sb_threshold`.

    The `bcg_pos` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    If no `position` is provided, the image centre is assumed.

    The purpose of this mask is to mask BCG flux, so that it is not included in an ICL measurement.
    No deblending is performed, the goal is to mask the entire region above the specified surface
    brightness threshold, centred on the BCG.
    """
    logger = logging.getLogger(__name__)
    logger.info("Creating BCG mask")
    segm = detect_sources(data=image, threshold=sb_threshold, npixels=5)
    segm = keep_segment_at_position(segm, image, centre_pos, wcs)
    bcg_mask = segm.make_source_mask()
    bcg_mask = binary_closing(bcg_mask, iterations=3)
    return bcg_mask

In [ ]:
# | export


def create_icl_mask(
    image: np.ndarray,  # input image data to mask
    *,  # the following parameters must be provided as keyword arguments if required
    nsigma=2.0,  # multiple of the background RMS required for detection
    smooth_sigma=10.0,  # sigma of the Gaussian used to smooth the image
    centre_pos=None,  # position of the cluster centre, if None assume the image centre
    bkg_box_size=500,  # the size (in pixels) of the boxes to use in estimating the background RMS
    bkg_filter_size=3,  # the size of the median filter to use in estimating the background RMS
    wcs=None,  # WCS for converting a `centre_pos` supplied as a SkyCoord to pixels
    dilation_radius=None,  # the radius of the circular structure used to dilate the mask
    median_filter=False,  # use a median filter to smooth the image
):  # the ICL image mask
    """Create an ICL mask.

    The background noise in the input `image` is estimated on a spatial scale of `bkg_box_size`,
    which is them multiplied by `nsigma` to define the detection threshold.

    The input `image` is smoothed using a Gaussian kernal of size `smooth_sigma` and sources
    detected above the defined threshold. Only the source at `centre_pos` is kept in the returned mask.
    The `centre_pos` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    If no `position` is provided, the image centre is assumed.

    The purpose of this mask is to mask ICL flux as completely as possible, so that it does not influence
    measurements of the background.
    """
    logger = logging.getLogger(__name__)
    logger.info("Creating ICL mask")
    with catch_warnings():
        filterwarnings("ignore", ".*Input data contains invalid values*")
        logger.debug(
            f"Estimating background with box size {bkg_box_size} and filter size {bkg_filter_size}"
        )
        bkg = get_background(
            image,
            box_size=bkg_box_size,
            filter_size=bkg_filter_size,
        )
    threshold = nsigma * bkg.background_rms
    logger.debug(f"Average {nsigma} sigma detection threshold: {threshold.mean():.5f}")

    logger.debug(f"Smoothing image with sigma={smooth_sigma}")
    if median_filter:
        logger.debug("Smoothing with a sampled median filter")
        smoothed_image = sampled_median_filter(
            image, size=smooth_sigma * 5, gaussian_sigma=smooth_sigma
        )
    else:
        logger.debug("Smoothing with a Gaussian kernel")
        smoothed_image = smooth_image(image, sigma=smooth_sigma)

    logger.debug("Detecting sources")
    segm = detect_sources(data=smoothed_image, threshold=threshold, npixels=5)

    logger.debug("Keeping segment at specified position")
    segm = keep_segment_at_position(segm, image, centre_pos, wcs)

    if dilation_radius is not None:
        logger.debug(f"Creating source mask with dilation radius {dilation_radius}")
        footprint = circular_footprint(dilation_radius)
    else:
        logger.debug("Creating source mask without dilation")
        footprint = None
    icl_mask = segm.make_source_mask(footprint=footprint)

    masked_percentage = np.sum(icl_mask) / icl_mask.size * 100
    logger.info(
        f"ICL mask created, {np.sum(icl_mask)} pixels masked ({masked_percentage:.2f}%)"
    )

    return icl_mask, smoothed_image

In [ ]:
# | export


def fast_mask(
    image: np.ndarray,  # input image data to mask
    *,  # the following parameters must be provided as keyword arguments if required
    mask_params=None,  # mask parameters
    estimate_background=False,  # estimate the background from the image median
    max_repeat=10,  # maximum number of masking iterations
    relative_rms_tolerance=0.001,  # relative tolerance in the rms required to stop iterating
    return_threshold=True,  # return the threshold
):
    logger = logging.getLogger(__name__)
    params = {
        "nsigma": 2.0,
        "smooth_sigma": 1.0,
        "background": 0.0,
        "erosion_iterations": 1,
        "dilation_radius": 2.0,
        "dilation_iterations": 2,
    }
    logger.info("Creating fast mask")
    if mask_params:
        params.update(mask_params)
    if params["smooth_sigma"] > 0:
        logger.info(f"smoothing image with sigma of {params['smooth_sigma']} pixels")
        smoothed_image = smooth_image(image, params["smooth_sigma"])
    else:
        smoothed_image = image
    bkg = params["background"]
    previous_rms = np.inf
    mask = None
    for i in range(max_repeat + 1):
        logger.info(f"iteration {i}")
        if mask is not None:
            valid_pixels = image[~mask & np.isfinite(image)]
        else:
            valid_pixels = image[np.isfinite(image)]
        if estimate_background:
            logger.debug(f"estimating background from {len(valid_pixels)} pixels")
            bkg = np.median(valid_pixels)
            logger.info(f"estimated background is {bkg}")
        rms = np.median(abs(valid_pixels - bkg)) / 0.67449
        logger.info(f"background RMS is {rms:.5f}")
        if abs(rms - previous_rms) / rms < relative_rms_tolerance:
            break
        previous_rms = rms
        # update the mask
        threshold = rms * params["nsigma"] + bkg
        logger.info(f"{params['nsigma']} sigma threshold is {threshold:.5f}")
        mask = smoothed_image > threshold
        if params["erosion_iterations"] > 0:
            logger.debug(f"eroding mask with {params['erosion_iterations']} iterations")
            mask = binary_erosion(mask, iterations=params["erosion_iterations"])
        if params["dilation_radius"] > 0 and params["dilation_iterations"] > 0:
            structure = circular_footprint(params["dilation_radius"])
            logger.debug(
                f"dilating mask with radius {params['dilation_radius']} pixels for {params['dilation_iterations']} iterations"
            )
            mask = binary_dilation(
                mask, structure=structure, iterations=params["dilation_iterations"]
            )
    if return_threshold:
        return mask, threshold
    else:
        return mask

In [ ]:
# | export


def dilated_object_mask(
    segm: np.ndarray,  # a photutils SegmentationImage
    growth: float = 0.5,  # the relative padding around each source
):
    """Create an object mask from a segmentation image.

    The mask is grown by repeatedly dilating the image, each time removing successively larger segments,
    such that large segments are dilated more.

    The dilation structure initially increases in size, so that resulting masks are relatively circular,
    while avoiding large structures that result in slow operations.

    Using `growth=0.5` increases masks by 50% of their original radius,
    while `growth=1.0` doubles their size.

    """
    logger = logging.getLogger(__name__)
    logger.info(f"Dilating object mask with growth factor of {growth}")
    segm = segm.copy()
    obj_mask = binary_dilation(segm.data > 0)
    if growth > 0:
        radius_min = 2
        while True:
            area_min = np.pi * radius_min**2
            logger.debug(f"current minimum size is {radius_min} pixels")
            small_labels = segm.labels[segm.areas < area_min]
            logger.debug(
                f"removing {len(small_labels)} segments smaller than {area_min:.2f} squared pixels"
            )
            segm.remove_labels(small_labels)
            logger.debug(f"{len(segm.labels)} segments remaining")
            if len(segm.labels) == 0:
                break
            growth_pixels = radius_min * growth
            radius = max(1, min(1, int(growth_pixels)))
            logger.debug(
                f"seeking to grow objects with radius {radius_min} pixels by {growth_pixels} pixels"
            )
            logger.debug(f"applying dilation with radius {radius} pixels")
            # radius = min(9, max(1, int(np.sqrt(growth_pixels))))
            # iterations = max(1, int(growth_pixels / radius))
            # logger.debug(
            #     f"seeking to grow objects with radius {radius_min} pixels by {growth_pixels} pixels"
            # )
            # logger.debug(
            #     f"applying {iterations} iterations of dilation with a radius of {radius}, equivalent to a growth of {iterations * radius} pixels"
            # )
            # big_mask = binary_dilation(
            #     segm.data > 0,
            #     structure=circular_footprint(radius),
            #     iterations=iterations,
            # )

            big_mask = segm.make_source_mask(footprint=circular_footprint(radius))
            logger.debug(f"new mask contains {big_mask.sum()} pixels")
            obj_mask = obj_mask | big_mask
            masked_percentage = obj_mask.sum() / obj_mask.size * 100
            logger.debug(
                f"currently {obj_mask.sum()} pixels masked ({masked_percentage:.2f}%)"
            )
            radius_min *= 2
    return obj_mask

In [ ]:
# | export


def create_object_mask(
    image: np.ndarray,  # input image data to mask
    *,  # the following parameters must be provided as keyword arguments if required
    exclude_mask=None,  # if provided, objects that overlap with this mask are removed
    exclude_position=False,  # segment at this position is removed; False: do not remove the segment, None: assume the image centre
    wcs=None,  # WCS for converting a `centre_pos` supplied as a SkyCoord to pixels
    growth=0.5,  # the relative padding around each source
    detection_params=None,  # detection parameters
):
    """Create an object mask."""
    logger = logging.getLogger(__name__)
    logger.info("Creating object mask")
    params = {
        "nsigma": 2.0,
        "background": 0.0,
        "smooth_sigma": 1.0,
        "npixels": 5,
        "nlevels": 32,
        "contrast": 0.01,
        "bkg_box_size": None,
        "bkg_filter_size": 1,
    }
    if detection_params:
        params.update(detection_params)
    logger.info("Estimating background and detection threshold")
    if params["bkg_box_size"] is not None:
        with catch_warnings():
            filterwarnings("ignore", ".*Input data contains invalid values*")
            bkg = get_background(
                image,
                box_size=params["bkg_box_size"],
                filter_size=params["bkg_filter_size"],
            )
        image = image - bkg.background
        threshold = bkg.background_rms * params["nsigma"] + params["background"]
    else:
        bkg = params["background"]
        threshold = detect_threshold(
            image,
            nsigma=params["nsigma"],
            background=bkg,
        )
    logger.info(f"Average {params['nsigma']} sigma threshold is {threshold.mean():.5f}")
    logger.info("Smoothing image")
    detection_image = smooth_image(image, sigma=params["smooth_sigma"])
    logger.info("Detecting sources")
    segm = detect_sources(
        data=detection_image,
        threshold=threshold,
        npixels=params["npixels"],
    )
    logger.info(f"Deblending {segm.nlabels} sources")
    with catch_warnings():
        filterwarnings("ignore", ".*[Dd]eblending mode.*changed.*")
        segm_deblend = deblend_sources(
            detection_image,
            segm,
            npixels=params["npixels"],
            nlevels=params["nlevels"],
            contrast=params["contrast"],
            progress_bar=False,
        )
    logger.info(f"Object mask contains {segm_deblend.nlabels} sources")
    segm_full = segm_deblend.copy()
    if exclude_mask is not None:
        logger.info(f"Supplied exclude mask contains {exclude_mask.sum()} pixels")
        segm_deblend.remove_masked_labels(exclude_mask, partial_overlap=True)
    elif exclude_position is not False:
        segm_deblend = remove_segment_at_position(
            segm_deblend, image, exclude_position, wcs
        )
    nlabels_centre = segm_full.nlabels - segm_deblend.nlabels
    if nlabels_centre > 0:
        logger.info(f"Removed {nlabels_centre} sources at the centre")
        logger.info(f"Object mask now contains {segm_deblend.nlabels} sources")
        excluded_labels = set(segm_full.labels) - set(segm_deblend.labels)
        segm_excluded = segm_full.copy()
        segm_excluded.keep_labels(list(excluded_labels))
        new_exclude_mask = segm_excluded.make_source_mask()
        if exclude_mask is not None:
            new_exclude_mask |= exclude_mask
        logger.info(f"New exclude mask contains {new_exclude_mask.sum()} pixels")
    else:
        new_exclude_mask = None
    logger.info("Dilating mask")
    obj_mask = dilated_object_mask(segm_deblend, growth)
    return obj_mask, bkg, threshold, new_exclude_mask

In [ ]:
# | export


def create_faint_mask(
    image: np.ndarray,  # input image data to mask
    *,  # the following parameters must be provided as keyword arguments if required
    exclude_mask=None,  # if provided, objects that are entirely covered by this mask are removed
    include_mask=None,  # if provided, only objects that overlap with this mask are kept
    exclude_position=False,  # segment at this position is removed; False: do not remove the segment, None: assume the image centre
    wcs=None,  # WCS for converting a `centre_pos` supplied as a SkyCoord to pixels
    growth=0.25,  # the relative padding around each source
    bkg_sigma=20,  # the sigma of the Gaussian weighted median filter used to estimate the background for detection
    detection_params=None,  # detection parameters
):
    """Create a faint mask by filtering the image before detection."""
    logger = logging.getLogger(__name__)
    params = {
        "nsigma": 2.0,
        "smooth_sigma": 1.0,
        "npixels": 5,
        "nlevels": 128,
        "contrast": 0.001,
    }
    if detection_params:
        params.update(detection_params)
    logger.info("Estimating background")
    bkg, rms = sampled_mad_filter(
        image.data, size=bkg_sigma * 5, nsample=None, gaussian_sigma=bkg_sigma
    )
    rms[~np.isfinite(rms)] = np.nanmean(rms)
    threshold = detect_threshold(
        image.data, nsigma=params["nsigma"], background=bkg, error=rms
    )
    logger.info(f"Average {params['nsigma']} sigma threshold is {threshold.mean():.5f}")
    logger.info("Smoothing image")
    detection_image = smooth_image(image, sigma=params["smooth_sigma"])
    if include_mask is not None:
        dilation_radius = 10
        dilation_iterations = int(
            round(np.sqrt(include_mask.sum() / np.pi) / dilation_radius)
        )
        logger.info(
            f"Creating detection mask by dilating include mask by {dilation_radius * dilation_iterations} pixels"
        )
        structure = circular_footprint(dilation_radius)
        detection_mask = binary_dilation(
            include_mask, structure=structure, iterations=dilation_iterations
        )
        detection_image[~detection_mask] = np.nan
    logger.info("Detecting sources")
    segm = detect_sources(
        data=detection_image,
        threshold=threshold,
        npixels=params["npixels"],
    )
    logger.info(f"Deblending {segm.nlabels} sources")
    with catch_warnings():
        filterwarnings("ignore", ".*[Dd]eblending mode.*changed.*")
        segm_deblend = deblend_sources(
            detection_image,
            segm,
            npixels=params["npixels"],
            nlevels=params["nlevels"],
            contrast=params["contrast"],
            progress_bar=False,
        )
    logger.info(f"Faint mask contains {segm_deblend.nlabels} sources")
    if include_mask is not None:
        logger.info("Keeping only objects that overlap with include_mask")
        segm_deblend.remove_masked_labels(~include_mask, partial_overlap=False)
        logger.info(f"Faint mask now contains {segm_deblend.nlabels} sources")
    if exclude_position is not False:
        logger.info("Removing segment at exclude_position")
        segm_deblend = remove_segment_at_position(
            segm_deblend, image, exclude_position, wcs
        )
        logger.info(f"Faint mask now contains {segm_deblend.nlabels} sources")
    if exclude_mask is not None:
        logger.info("Removing objects entirely covered by exclude_mask")
        segm_deblend.remove_masked_labels(exclude_mask, partial_overlap=False)
        logger.info(f"Faint mask now contains {segm_deblend.nlabels} sources")
    logger.info("Dilating mask")
    faint_mask = dilated_object_mask(segm_deblend, growth)
    return faint_mask, bkg, threshold

In [ ]:
# | export


def plot_mask(
    img,  # image to plot
    mask,  # mask to plot
    *,  # the following parameters must be provided as keyword arguments if required
    detail=False,  # if True, add a row of detail images
    detail_size=1000,  # size, in pixels, of the detail region
    background_focus=False,
    show_mask=True,
):
    """Create a plot displaying the original image, masked image and mask."""
    nrows = 2 if detail else 1
    ncols = 3 if show_mask else 2
    fw, fh = plt.rcParams["figure.figsize"]
    fw = 2 * fw
    fh = fw * nrows / ncols
    fig, ax = plt.subplots(
        nrows, ncols, figsize=(fw, fh), squeeze=False, layout="constrained"
    )
    rms = median_abs_deviation(img, axis=None, scale="normal", nan_policy="omit")
    median = np.nanmedian(img)
    if background_focus:
        norm = ImageNormalize(vmin=median - 2.5 * rms, vmax=median + 2.5 * rms)
    else:
        norm = ImageNormalize(
            vmin=median - 2.5 * rms, vmax=median + 10 * rms, stretch=AsinhStretch(0.1)
        )
    cmap = mpl.colormaps.get_cmap("viridis")
    cmap.set_bad("black")
    ax[0, 0].imshow(img, norm=norm, origin="lower", interpolation="none", cmap=cmap)
    ax[0, 1].imshow(
        np.where(mask, np.nan, img),
        norm=norm,
        origin="lower",
        interpolation="none",
        cmap=cmap,
    )
    ax[0, 0].set_title("Original Image")
    ax[0, 1].set_title("Masked Image")
    if show_mask:
        ax[0, 2].imshow(mask, origin="lower", interpolation="none", cmap="gray_r")
        ax[0, 2].set_title("Object Mask")
    if detail:
        ci, cj = get_img_centre_pixel(img).astype(int)
        ax[0, 0].plot(
            [ci - detail_size, ci, ci, ci - detail_size, ci - detail_size],
            [cj, cj, cj - detail_size, cj - detail_size, cj],
            "w-",
        )
        ax[0, 0].set_ylabel("Full Image")
        ax[1, 0].set_ylabel("Detail")
        img = img[ci - detail_size : ci, cj - detail_size : cj]
        mask = mask[ci - detail_size : ci, cj - detail_size : cj]
        ax[1, 0].imshow(img, norm=norm, origin="lower", interpolation="none", cmap=cmap)
        ax[1, 1].imshow(
            np.where(mask, np.nan, img),
            norm=norm,
            origin="lower",
            interpolation="none",
            cmap=cmap,
        )
        if show_mask:
            ax[1, 2].imshow(mask, origin="lower", interpolation="none", cmap="gray_r")
    for a in ax.flat:
        a.xaxis.set_ticks([])
        a.yaxis.set_ticks([])

In [ ]:
# | export


def write_mask(
    image_filename,  # full filename of the image to which the mask corresponds
    mask_name,  # name of the mask
    mask_data,  # the mask data to write
):
    """Save `mask_data` as a FITS file, at the same `image_filename` but with `name` appended."""
    fn = image_filename.replace(".fits", f"_{mask_name}.fits")
    fits.writeto(fn, mask_data.astype("uint8"), overwrite=True)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()